<a href="https://colab.research.google.com/github/Andre-Sil/Andre-Sil/blob/main/b3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# %% [markdown]
# # 📈 GEBRA Portfolio v16.2 – ARQUITETURA COMPLETA COM ESTRUTURA DE FUNDO
#
# **Fidelidade ao material original:**
# - Estrutura de Fundo (Aulas 11, 35, 68-71)
# - Sensibilidade Estrutural do Canal (Aula 24)
# - Teoria de Dow (Aula 9-12)
# - Polaridade de Suportes (Aula 35)
# - Confluência mínima de 3 sinais independentes (Aula 8)
#
# **Novas camadas:**
# - Detecção de estrutura de fundo (OCO Invertido, Fundo Duplo/Triplo) acima de LTB rompida
# - Penalização de vendas contra rompimento macro
# - Bonificação de compras em pullback pós-rompimento
# - Log detalhado com watchlist de setups fortes

# %% code
import os

# ==================== CONFIGURAÇÕES DE ACESSO ====================
EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
BRAPI_TOKEN = os.getenv('BRAPI_API_TOKEN', '') or os.getenv('BRAPI_TOKEN', '')
TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN', '')
TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID', '')

# ==================== PARÂMETROS GLOBAIS ====================
CAPITAL_TOTAL = 100000.0
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 2.00
MAX_SETUPS_POR_DIA = 5
MAX_ATIVOS_POR_SETOR = 2
PAYOFF_MINIMO = 3.0

# --- Parâmetros de padrões ---
TOLERANCIA_NIVEL = 0.04
TOLERANCIA_OMBRO = 0.08
CORPO_MINIMO_CANDLE = 0.50
FECHAMENTO_EXTREMIDADE = 0.25
VOLUME_MULT_ALTO = 2.0
VOLUME_MULT_MEDIO_COMPRA = 1.2
VOLUME_MULT_VENDA = 0.5
ATR_PERIODOS = 14
VOLUME_FORMACAO_MAX_MEDIA = 0.5
CUP_TOPO_CORRECAO_MAX = 0.50
CUP_HANDLE_MAX_DIAS = 14
PIVO_ORDEM = 3

# --- Confluência ---
PONTUACAO_MINIMA_CONFLUENCIA = 60
MIN_SINAIS_CONFLUENCIA = 3
IFR_MAX_COMPRA = 70
IFR_MIN_COMPRA = 25
IFR_MAX_VENDA = 75
IFR_MIN_VENDA = 30

# --- Filtros de liquidez e dados ---
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
ADX_MINIMO = 25
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
FALLBACK_TICKERS = ['PETR4','VALE3','ITUB4','BBDC4','BBAS3','ABEV3','WEGE3','RADL3','SUZB3','GGBR4','MGLU3','VVAR3','RENT3','RAIL3','CCRO3','ELET3','CPFE3','SBSP3','SANB11','B3SA3','JBSS3','BRFS3','KLBN11','EQTL3']
TICKERS_BLOQUEADOS = ['GFSA3.SA','ONCO3.SA','PMAM3.SA','AZTE3.SA','RAIZ4.SA','BHIA3.SA','CASH3.SA','LJQQ3.SA','RCSL4.SA','HBOR3.SA']

# --- Timeframes ---
ANALISAR_DIARIO = True
ANALISAR_SEMANAL = True
ANALISAR_MENSAL = True

# --- Modo teste ---
MODO_TESTE = False
TESTE_TICKERS = ['PETR4','VALE3','ITUB4','BBDC4','BBAS3','ABEV3','WEGE3','RADL3','GGBR4','MGLU3']

# --- Logging ---
ARQUIVO_LOG = "trading_log_v16_2.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v16_2.log"

# --- Feature flags ---
MODO_PULLBACK = True
USAR_OBV = True
USAR_DIVERGENCIAS = True
USAR_GAPS = True
USAR_BOLLINGER = True
USAR_MACD_CONFLUENCIA = True
USAR_FILTRO_MACRO_LTB = True
USAR_ESTRUTURA_FUNDO = True

# --- Thresholds ---
GAP_MIN_ATR_MULT = 0.5
GAP_VOLUME_MULT = 1.5
GAP_EXAUSTAO_VOLUME_MULT = 2.0
BOLLINGER_SQUEEZE_LOOKBACK = 120
BOLLINGER_SQUEEZE_TOL = 1.1
PULLBACK_TOLERANCIA_PCT = 0.01
PULLBACK_VOLUME_MAX_PCT = 0.8
LTB_JANELA = 120

print("✅ Parâmetros v16.2 carregados – Estrutura de Fundo ativada")

✅ Parâmetros v16.2 carregados – Estrutura de Fundo ativada


In [22]:
# %% code
# ================ INSTALAÇÃO E IMPORTAÇÕES ================
!pip install yfinance pandas-ta --quiet 2>/dev/null
!pip install brapi

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, sys, traceback, gc, socket
from collections import Counter
from brapi import Brapi

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

try:
    from scipy.signal import argrelextrema
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("⚠️ scipy não disponível. Usando detecção manual de pivôs.")

print("✅ Bibliotecas carregadas")

⚠️ scipy não disponível. Usando detecção manual de pivôs.
✅ Bibliotecas carregadas


In [23]:
# %% code
# ================ LOGGER E UTILITÁRIOS ================
class Logger:
    def __init__(self, log_json, log_detalhado):
        self.log_json = log_json
        self.log_detalhado = log_detalhado
        self.t0 = time.time()
        self.tm = {}
        self.buffer = []
        self.max_buffer = 100

    def log(self, msg, nivel="INFO", extra=None):
        ts = datetime.now().strftime("%H:%M:%S")
        linha = f"[{ts}] [{nivel}] {msg}"
        if extra:
            linha += f" | {extra}"
        print(linha)
        if self.log_detalhado:
            self.buffer.append(linha + "\n")
            if len(self.buffer) >= self.max_buffer:
                self._flush()

    def _flush(self):
        if self.log_detalhado and self.buffer:
            try:
                with open(self.log_detalhado, 'a', encoding='utf-8') as f:
                    f.writelines(self.buffer)
                self.buffer.clear()
            except Exception as e:
                print(f"Erro ao gravar log: {e}")

    def warn(self, msg, extra=None):
        self.log(msg, "WARN", extra)

    def error(self, msg, extra=None):
        self.log(msg, "ERRO", extra)

    def inicio(self, etapa):
        self.tm[etapa] = {'ini': time.time()}
        self.log(f"🚀 INÍCIO: {etapa}", "ETAPA")

    def fim(self, etapa, dados=None):
        if etapa in self.tm:
            dur = time.time() - self.tm[etapa]['ini']
            self.tm[etapa]['dur'] = dur
            msg = f"✅ FIM: {etapa} ({dur:.1f}s)"
            if dados:
                msg += " | " + " | ".join(f"{k}:{v}" for k, v in dados.items())
            self.log(msg, "ETAPA")

    def resumo(self):
        self._flush()
        total = time.time() - self.t0
        self.log("\n" + "="*60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.1f}s", "RESUMO")
        for etapa, dados in self.tm.items():
            if 'dur' in dados:
                pct = dados['dur']/total*100 if total>0 else 0
                self.log(f"   • {etapa}: {dados['dur']:.1f}s ({pct:.0f}%)", "RESUMO")
        self.log("="*60 + "\n", "RESUMO")

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)

def _log_exc(contexto, e):
    try:
        msg = f"[{contexto}] {type(e).__name__}: {str(e)[:200]}"
        logger.error(msg)
        with open('traceback_errors.log', 'a', encoding='utf-8') as f:
            f.write(f"\n{'='*60}\n{datetime.now()}\n{contexto}\n{str(e)}\n{traceback.format_exc()}")
    except:
        pass

def enviar_telegram(mensagem, parse_mode='HTML'):
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID:
        return
    try:
        requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
                      data={'chat_id': TELEGRAM_CHAT_ID, 'text': mensagem, 'parse_mode': parse_mode}, timeout=10)
    except Exception as e:
        _log_exc('Telegram', e)

def verificar_conectividade():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

print("✅ Logger e utilitários carregados")

✅ Logger e utilitários carregados


In [24]:
# %% code
# ================ DOWNLOAD DE DADOS (BRAPI PRIMÁRIA + YFINANCE) ================
def baixar_dados_brapi_otimizado(tickers_sa, periodo_anos=5, interval='1d'):
    """
    Baixa dados históricos usando a SDK oficial brapi.
    Lê o token da variável BRAPI_TOKEN (já definida via BRAPI_API_TOKEN).
    """
    if not BRAPI_TOKEN:
        logger.warn("BRAPI_TOKEN não configurado. Usando modo público (apenas tickers de teste).")
        client = Brapi()
    else:
        client = Brapi(api_key=BRAPI_TOKEN)

    resultados = {}
    tickers_limpos = [t.replace('.SA', '') for t in tickers_sa]
    lote_tamanho = 20

    for i in range(0, len(tickers_limpos), lote_tamanho):
        lote = tickers_limpos[i:i+lote_tamanho]
        tickers_str = ','.join(lote)
        logger.log(f"📡 Brapi SDK lote {i//lote_tamanho + 1}: {tickers_str[:80]}...")

        try:
            resp = client.quote.retrieve(
                tickers=tickers_str,
                range=f'{periodo_anos}y',
                interval=interval
            )
            quotes = resp.results if hasattr(resp, 'results') else resp
            if not isinstance(quotes, list):
                quotes = [quotes]

            for q in quotes:
                tk = q.symbol + '.SA'
                hist = q.historical_data_price if hasattr(q, 'historical_data_price') else []
                if hist and len(hist) > 0:
                    df = pd.DataFrame(hist)
                    df['date'] = pd.to_datetime(df['date'], unit='s')
                    df.set_index('date', inplace=True)
                    df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'}, inplace=True)
                    df = df[['Open','High','Low','Close','Volume']]
                    resultados[tk] = df
        except Exception as e:
            _log_exc(f'Brapi SDK lote {tickers_str[:30]}', e)
            time.sleep(2)
        time.sleep(0.5)

    logger.log(f"📦 Brapi SDK: {len(resultados)} tickers")
    return resultados

def baixar_dados_yfinance_v3(tickers_sa, periodo='5y', max_tentativas=5):
    data = {}
    for i, t in enumerate(tickers_sa):
        if i % 20 == 0: logger.log(f"🔄 yfinance v3: {i+1}/{len(tickers_sa)}")
        df = None
        for tentativa in range(max_tentativas):
            try:
                df = yf.Ticker(t).history(period=periodo, auto_adjust=True)
                if df is not None and not df.empty and 'Close' in df.columns: break
                time.sleep(0.5 * (tentativa + 1))
            except Exception as e:
                if tentativa == max_tentativas - 1: _log_exc(f'yfinance v3 {t}', e)
                time.sleep(0.5 * (tentativa + 1))
        if df is None or df.empty:
            try:
                t_sem_sa = t.replace('.SA', '')
                df = yf.Ticker(t_sem_sa).history(period=periodo, auto_adjust=True)
                if df is not None and not df.empty and 'Close' in df.columns:
                    logger.log(f"✅ {t} recuperado sem .SA", nivel="WARN")
            except Exception: pass
        if df is not None and not df.empty and 'Close' in df.columns:
            df = df[['Open','High','Low','Close','Volume']].copy()
            data[t] = df
        time.sleep(0.3)
    logger.log(f"📦 yfinance v3: {len(data)} tickers")
    return data

def obter_tickers_brapi():
    if not BRAPI_TOKEN: return []
    tickers = []
    try:
        url = "https://brapi.dev/api/quote/list"
        headers = {'Authorization': f'Bearer {BRAPI_TOKEN}'}
        page = 1
        while True:
            params = {'limit': 100, 'page': page, 'type': 'stock'}
            resp = requests.get(url, headers=headers, params=params, timeout=15)
            if resp.status_code != 200: break
            data = resp.json()
            stocks = data.get('stocks', [])
            if not stocks: break
            for s in stocks: tickers.append(s['stock'])
            page += 1
            time.sleep(0.2)
        logger.log(f"📋 BRAPI list: {len(tickers)} tickers")
    except Exception as e:
        logger.warn(f"Erro ao obter lista da brapi: {e}")
    return tickers

def obter_tickers_scraping():
    try:
        resp = requests.get("https://www.dadosdemercado.com.br/acoes", timeout=10, headers={'User-Agent':'Mozilla/5.0'})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cells = row.find_all('td')
            if cells and not cells[0].text.strip().startswith('#'): tickers.append(cells[0].text.strip().replace('.SA', ''))
        return tickers
    except Exception as e:
        logger.warn(f"Scraping falhou: {e}")
        return []

def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE) as f: cache = json.load(f)
            if (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600 < 24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)")
                return cache['tickers']
        except: pass
    tickers = []
    if BRAPI_TOKEN: tickers = obter_tickers_brapi()
    if not tickers: tickers = obter_tickers_scraping()
    if not tickers:
        tickers = FALLBACK_TICKERS.copy()
        logger.warn("Usando lista fallback de tickers")
    if tickers:
        with open(CACHE_TICKERS_FILE, 'w') as f:
            json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f)
    return tickers

print("✅ Módulo de download carregado (Brapi primária)")

✅ Módulo de download carregado (Brapi primária)


In [25]:
# %% code
# ================ FUNÇÕES AUXILIARES ================
def encontrar_picos(series, ordem=PIVO_ORDEM, modo='max'):
    if SCIPY_AVAILABLE:
        if modo == 'max':
            indices = argrelextrema(series.values, np.greater, order=ordem)[0]
        else:
            indices = argrelextrema(series.values, np.less, order=ordem)[0]
        return [(int(i), series.iloc[i]) for i in indices if i > ordem and i < len(series)-ordem]
    else:
        pivos = []
        for i in range(ordem, len(series)-ordem):
            if modo == 'max' and series.iloc[i] == max(series.iloc[i-ordem:i+ordem+1]): pivos.append((i, series.iloc[i]))
            elif modo == 'min' and series.iloc[i] == min(series.iloc[i-ordem:i+ordem+1]): pivos.append((i, series.iloc[i]))
        return pivos

def calcular_atr(df, periodo=ATR_PERIODOS):
    try:
        atr_serie = ta.atr(df['High'], df['Low'], df['Close'], length=periodo)
        return float(atr_serie.iloc[-1]) if atr_serie is not None and not atr_serie.empty else 0.0
    except: return 0.0

def calcular_fibonacci_retracao(df, n_barras=50):
    if len(df) < n_barras: return None
    highs = df['High']; lows = df['Low']
    max_idx = highs.rolling(n_barras).max().idxmax()
    min_idx = lows.rolling(n_barras).min().idxmin()
    if max_idx < min_idx:
        topo = highs.loc[max_idx:min_idx].max()
        fundo = lows.loc[max_idx:min_idx].min()
    else:
        topo = highs.loc[min_idx:max_idx].max()
        fundo = lows.loc[min_idx:max_idx].min()
    if topo <= fundo or fundo <= 0: return None
    diff = topo - fundo
    return {'38.2%': round(topo - diff*0.382, 2), '50.0%': round(topo - diff*0.5, 2), '61.8%': round(topo - diff*0.618, 2)}

def preco_em_zona_interesse(df, preco, direcao, tolerancia=0.02):
    suportes = [v for _, v in encontrar_picos(df['Low'], modo='min')[-10:]]
    resistencias = [v for _, v in encontrar_picos(df['High'], modo='max')[-10:]]
    fib = calcular_fibonacci_retracao(df)
    niveis = []
    if fib: niveis.extend([fib['38.2%'], fib['50.0%'], fib['61.8%']])
    if direcao == 'COMPRA':
        for sup in suportes:
            if abs(preco - sup) / sup <= tolerancia: return True
        for niv in niveis:
            if abs(preco - niv) / niv <= tolerancia: return True
    else:
        for res in resistencias:
            if abs(res - preco) / preco <= tolerancia: return True
        for niv in niveis:
            if abs(niv - preco) / preco <= tolerancia: return True
    return False

print("✅ Funções auxiliares carregadas")

✅ Funções auxiliares carregadas


In [26]:
# %% code
# ================ PADRÕES DE VELAS (ESTENDIDOS) ================
def detectar_martelo(row, direcao='COMPRA'):
    corpo = abs(row['Close'] - row['Open']); range_total = row['High'] - row['Low']
    if range_total == 0: return False
    if corpo > range_total * 0.35: return False
    if direcao == 'COMPRA':
        sombra_inf = min(row['Open'], row['Close']) - row['Low']
        sombra_sup = row['High'] - max(row['Open'], row['Close'])
        return sombra_inf >= 2 * corpo and sombra_sup <= corpo * 0.5
    return False

def detectar_estrela_cadente(row):
    corpo = abs(row['Close'] - row['Open']); range_total = row['High'] - row['Low']
    if range_total == 0: return False
    sombra_sup = row['High'] - max(row['Open'], row['Close'])
    sombra_inf = min(row['Open'], row['Close']) - row['Low']
    return (sombra_sup >= 2 * corpo and corpo <= range_total * 0.35 and sombra_inf <= corpo * 0.5)

def detectar_doji(row):
    corpo = abs(row['Close'] - row['Open']); range_total = row['High'] - row['Low']
    if range_total == 0: return True
    return corpo <= range_total * 0.05

def detectar_engolfo_compra(row, row_ant):
    if row_ant is None: return False
    corpo_atual = abs(row['Close'] - row['Open']); corpo_ant = abs(row_ant['Close'] - row_ant['Open'])
    if corpo_ant == 0: return False
    return (row['Close'] > row['Open'] and row_ant['Close'] < row_ant['Open'] and
            row['Open'] <= row_ant['Close'] and row['Close'] >= row_ant['Open'])

def detectar_engolfo_baixa(row, row_ant):
    if row_ant is None: return False
    corpo_atual = abs(row['Close'] - row['Open']); corpo_ant = abs(row_ant['Close'] - row_ant['Open'])
    if corpo_ant == 0: return False
    return (row['Close'] < row['Open'] and row_ant['Close'] > row_ant['Open'] and
            row['Open'] >= row_ant['Close'] and row['Close'] <= row_ant['Open'])

def detectar_kicker_baixa(row, row_ant):
    if row_ant is None: return False
    corpo = abs(row['Close'] - row['Open']); range_total = row['High'] - row['Low']
    if range_total == 0: return False
    return (row['Open'] < row_ant['Low'] and row['Close'] < row['Open'] and corpo / range_total >= 0.7)

def validar_candle_forca_v2(row, direcao='COMPRA'):
    rng = row['High'] - row['Low']
    if rng <= 0: return False, "range zero"
    corpo = abs(row['Close'] - row['Open'])
    if corpo / rng < CORPO_MINIMO_CANDLE: return False, f"corpo/range = {corpo/rng:.2f} < {CORPO_MINIMO_CANDLE}"
    if direcao == 'COMPRA':
        fech_rel = (row['Close'] - row['Low']) / rng
        if fech_rel < (1 - FECHAMENTO_EXTREMIDADE): return False, f"fechamento a {fech_rel:.0%} da mínima"
    else:
        fech_rel = (row['High'] - row['Close']) / rng
        if fech_rel < (1 - FECHAMENTO_EXTREMIDADE): return False, f"fechamento a {fech_rel:.0%} da máxima"
    return True, "OK"

def classificar_candle(row, row_anterior, direcao):
    padroes = []
    if direcao == 'COMPRA':
        if detectar_martelo(row, 'COMPRA'): padroes.append('Martelo')
        if row_anterior is not None and detectar_engolfo_compra(row, row_anterior): padroes.append('Engolfo de Alta')
    else:
        if detectar_estrela_cadente(row): padroes.append('Estrela Cadente')
        if row_anterior is not None and detectar_engolfo_baixa(row, row_anterior): padroes.append('Engolfo de Baixa')
        if row_anterior is not None and detectar_kicker_baixa(row, row_anterior): padroes.append('Kicker de Baixa')
    if detectar_doji(row): padroes.append('Doji')
    corpo = abs(row['Close'] - row['Open']); rng = row['High'] - row['Low']
    if rng > 0 and corpo / rng >= 0.8: padroes.append('Corpo Longo')
    return padroes

print("✅ Detectores de velas carregados")

✅ Detectores de velas carregados


In [27]:
# %% code
# ================ OBV E DIVERGÊNCIAS ================
def calcular_obv(df):
    obv = [0]
    for i in range(1, len(df)):
        if df['Close'].iloc[i] > df['Close'].iloc[i-1]: obv.append(obv[-1] + df['Volume'].iloc[i])
        elif df['Close'].iloc[i] < df['Close'].iloc[i-1]: obv.append(obv[-1] - df['Volume'].iloc[i])
        else: obv.append(obv[-1])
    return pd.Series(obv, index=df.index)

def obv_antecipacao(df, direcao):
    if len(df) < 50: return False
    obv = calcular_obv(df)
    if direcao == 'COMPRA':
        topos_preco = encontrar_picos(df['High'], modo='max')
        if len(topos_preco) < 1: return False
        idx_topo, valor_topo = topos_preco[-1]
        obv_topo = obv.iloc[idx_topo]
        return obv.iloc[-1] > obv_topo and df['Close'].iloc[-1] < valor_topo
    else:
        fundos_preco = encontrar_picos(df['Low'], modo='min')
        if len(fundos_preco) < 1: return False
        idx_fundo, valor_fundo = fundos_preco[-1]
        obv_fundo = obv.iloc[idx_fundo]
        return obv.iloc[-1] < obv_fundo and df['Close'].iloc[-1] > valor_fundo

def detectar_divergencia_oscilador(df, oscilador='IFR', direcao='COMPRA'):
    if len(df) < 50: return False
    if oscilador == 'IFR': serie_osc = ta.rsi(df['Close'], length=14)
    elif oscilador == 'MACD': serie_osc = ta.macd(df['Close'], fast=12, slow=26, signal=9)['MACD_12_26_9']
    else: return False
    if direcao == 'COMPRA':
        fundos_preco = encontrar_picos(df['Low'], modo='min')
        if len(fundos_preco) < 2: return False
        f1, f2 = fundos_preco[-2], fundos_preco[-1]
        if f2[1] < f1[1]:
            osc1, osc2 = serie_osc.iloc[f1[0]] if f1[0] < len(serie_osc) else np.nan, serie_osc.iloc[f2[0]] if f2[0] < len(serie_osc) else np.nan
            if pd.notna(osc1) and pd.notna(osc2) and osc2 > osc1: return True
    else:
        topos_preco = encontrar_picos(df['High'], modo='max')
        if len(topos_preco) < 2: return False
        t1, t2 = topos_preco[-2], topos_preco[-1]
        if t2[1] > t1[1]:
            osc1, osc2 = serie_osc.iloc[t1[0]] if t1[0] < len(serie_osc) else np.nan, serie_osc.iloc[t2[0]] if t2[0] < len(serie_osc) else np.nan
            if pd.notna(osc1) and pd.notna(osc2) and osc2 < osc1: return True
    return False

print("✅ OBV e Divergências carregados")

✅ OBV e Divergências carregados


In [28]:
# %% code
# ================ GAPS ================
def detectar_gap(df, i=-1):
    if len(df) < abs(i)+2: return None
    row, row_ant = df.iloc[i], df.iloc[i-1]
    if row['Low'] > row_ant['High']: return ('ALTA', (row['Low'] - row_ant['High']) / row_ant['High'])
    elif row['High'] < row_ant['Low']: return ('BAIXA', (row_ant['Low'] - row['High']) / row_ant['Low'])
    return None

def classificar_gap(df, atr, media_volume):
    gap_info = detectar_gap(df)
    if gap_info is None: return None
    tipo, tamanho = gap_info
    if tamanho * df['Close'].iloc[-1] < GAP_MIN_ATR_MULT * atr: return None
    vol = df['Volume'].iloc[-1]
    if vol > media_volume * GAP_VOLUME_MULT: return 'Fuga'
    row = df.iloc[-1]
    corpo = abs(row['Close'] - row['Open']); rng = row['High'] - row['Low']
    corpo_pequeno = (rng > 0) and (corpo / rng < 0.5)
    mme20 = df['Close'].ewm(span=20).mean()
    acima_mme = (df['Close'] > mme20).tail(15).sum()
    if vol > media_volume * GAP_EXAUSTAO_VOLUME_MULT and corpo_pequeno and acima_mme >= 15: return 'Exaustão'
    return 'Comum'

print("✅ Detector de Gaps carregado")


✅ Detector de Gaps carregado


In [29]:
# %% code
# ================ FIBONACCI ALVOS ================
def calcular_expansao_fibonacci(df, direcao):
    if len(df) < 30: return None
    if direcao == 'COMPRA':
        fundos = encontrar_picos(df['Low'], modo='min')
        topos = encontrar_picos(df['High'], modo='max')
        if len(fundos) < 2 or len(topos) < 1: return None
        fundo1 = fundos[-2][1]
        topo = max([t[1] for t in topos if t[0] > fundos[-2][0]] + [fundos[-1][1]])
        fundo2 = fundos[-1][1]
        diff = topo - fundo1
        if diff <= 0: return None
        return {'161.8%': round(fundo2 + diff * 1.618, 2), '261.8%': round(fundo2 + diff * 2.618, 2)}
    else:
        topos = encontrar_picos(df['High'], modo='max')
        fundos = encontrar_picos(df['Low'], modo='min')
        if len(topos) < 2 or len(fundos) < 1: return None
        topo1 = topos[-2][1]
        fundo = min([f[1] for f in fundos if f[0] > topos[-2][0]] + [topos[-1][1]])
        topo2 = topos[-1][1]
        diff = topo1 - fundo
        if diff <= 0: return None
        return {'161.8%': round(topo2 - diff * 1.618, 2), '261.8%': round(topo2 - diff * 2.618, 2)}

print("✅ Fibonacci alvos carregado")

✅ Fibonacci alvos carregado


In [30]:
# %% code
# ================ BOLLINGER BANDS ================
def calcular_bollinger_squeeze(df, lookback=BOLLINGER_SQUEEZE_LOOKBACK, tol=BOLLINGER_SQUEEZE_TOL):
    if len(df) < lookback: return False
    bbands = ta.bbands(df['Close'], length=20, std=2)
    if bbands is None: return False
    col_sup = [c for c in bbands.columns if c.startswith('BBU')][0]
    col_inf = [c for c in bbands.columns if c.startswith('BBL')][0]
    col_media = [c for c in bbands.columns if c.startswith('BBM')][0]
    banda_sup, banda_inf, banda_media = bbands[col_sup], bbands[col_inf], bbands[col_media]
    largura_atual = (banda_sup.iloc[-1] - banda_inf.iloc[-1]) / banda_media.iloc[-1]
    historico_largura = (banda_sup - banda_inf) / banda_media
    minimo_historico = historico_largura.rolling(lookback).min().iloc[-1]
    return largura_atual <= minimo_historico * tol

print("✅ Bollinger Bands carregado")

✅ Bollinger Bands carregado


In [31]:
# %% code
# ================ PULLBACK ================
def verificar_pullback(df, linha_rompida, direcao, media_volume):
    if len(df) < 3: return False, None
    for i in range(-5, 0):
        row = df.iloc[i]
        if direcao == 'COMPRA':
            if row['Low'] <= linha_rompida * (1 + PULLBACK_TOLERANCIA_PCT) and row['Low'] >= linha_rompida * (1 - PULLBACK_TOLERANCIA_PCT):
                if df['Volume'].iloc[i] < media_volume * PULLBACK_VOLUME_MAX_PCT:
                    row_ant = df.iloc[i-1] if abs(i) < len(df) else None
                    if detectar_martelo(row, 'COMPRA') or (row_ant is not None and detectar_engolfo_compra(row, row_ant)): return True, row
        else:
            if row['High'] <= linha_rompida * (1 + PULLBACK_TOLERANCIA_PCT) and row['High'] >= linha_rompida * (1 - PULLBACK_TOLERANCIA_PCT):
                if df['Volume'].iloc[i] < media_volume * PULLBACK_VOLUME_MAX_PCT:
                    row_ant = df.iloc[i-1] if abs(i) < len(df) else None
                    if detectar_estrela_cadente(row) or (row_ant is not None and detectar_engolfo_baixa(row, row_ant)): return True, row
    return False, None

print("✅ Detector de Pullback carregado")


✅ Detector de Pullback carregado


In [32]:
# %% code
# ================ PADRÕES GRÁFICOS ================
def _mesmo_nivel(vals, tol=TOLERANCIA_NIVEL):
    if len(vals) < 2: return True
    ref = vals[0]
    return all(abs(v - ref) / ref <= tol for v in vals)

def detectar_triangulo_simetrico(df, min_pontos=2):
    if len(df) < 30: return False, {}
    highs = encontrar_picos(df['High'], modo='max')[-min_pontos*2:]
    lows = encontrar_picos(df['Low'], modo='min')[-min_pontos*2:]
    if len(highs) < min_pontos or len(lows) < min_pontos: return False, {}
    topos = [v for _, v in highs[-min_pontos:]]
    fundos = [v for _, v in lows[-min_pontos:]]
    if all(topos[i] > topos[i+1] for i in range(len(topos)-1)) and all(fundos[i] < fundos[i+1] for i in range(len(fundos)-1)):
        return True, {'tipo': 'Triângulo Simétrico', 'altura': max(topos) - min(fundos)}
    return False, {}

def detectar_triangulo_ascendente(df, min_pontos=2):
    if len(df) < 30: return False, {}
    highs = encontrar_picos(df['High'], modo='max')[-min_pontos:]
    lows = encontrar_picos(df['Low'], modo='min')[-min_pontos*2:]
    if len(highs) < min_pontos or len(lows) < min_pontos: return False, {}
    topos = [v for _, v in highs]
    fundos = [v for _, v in lows[-min_pontos:]]
    if _mesmo_nivel(topos) and all(fundos[i] < fundos[i+1] for i in range(len(fundos)-1)):
        return True, {'tipo': 'Triângulo Ascendente', 'altura': max(topos) - min(fundos), 'resistencia': topos[0]}
    return False, {}

def detectar_triangulo_descendente(df, min_pontos=2):
    if len(df) < 30: return False, {}
    highs = encontrar_picos(df['High'], modo='max')[-min_pontos*2:]
    lows = encontrar_picos(df['Low'], modo='min')[-min_pontos:]
    if len(highs) < min_pontos or len(lows) < min_pontos: return False, {}
    topos = [v for _, v in highs[-min_pontos:]]
    fundos = [v for _, v in lows]
    if _mesmo_nivel(fundos) and all(topos[i] > topos[i+1] for i in range(len(topos)-1)):
        return True, {'tipo': 'Triângulo Descendente', 'altura': max(topos) - min(fundos), 'suporte': fundos[0]}
    return False, {}

def detectar_retangulo(df, min_toques=2):
    if len(df) < 20: return False, {}
    highs = encontrar_picos(df['High'], modo='max')[-min_toques:]
    lows = encontrar_picos(df['Low'], modo='min')[-min_toques:]
    if len(highs) < min_toques or len(lows) < min_toques: return False, {}
    topos = [v for _, v in highs]; fundos = [v for _, v in lows]
    if _mesmo_nivel(topos) and _mesmo_nivel(fundos):
        return True, {'tipo': 'Retângulo', 'altura': topos[0] - fundos[0], 'resistencia': topos[0], 'suporte': fundos[0]}
    return False, {}

def detectar_oco(df, tipo='topo'):
    if len(df) < 50: return False, {}
    if tipo == 'topo':
        picos = encontrar_picos(df['High'], modo='max')
        if len(picos) < 3: return False, {}
        ombro_e, cabeca, ombro_d = picos[-3], picos[-2], picos[-1]
        if not (cabeca[1] > ombro_e[1] and cabeca[1] > ombro_d[1]): return False, {}
        if abs(ombro_e[1] - ombro_d[1]) / ombro_e[1] > TOLERANCIA_OMBRO: return False, {}
        vale1 = min(df['Low'].iloc[ombro_e[0]:cabeca[0]])
        vale2 = min(df['Low'].iloc[cabeca[0]:ombro_d[0]])
        neckline = (vale1 + vale2) / 2
        return True, {'tipo': 'OCO Topo', 'altura': cabeca[1] - neckline, 'neckline': neckline}
    else:
        vales = encontrar_picos(df['Low'], modo='min')
        if len(vales) < 3: return False, {}
        ombro_e, cabeca, ombro_d = vales[-3], vales[-2], vales[-1]
        if not (cabeca[1] < ombro_e[1] and cabeca[1] < ombro_d[1]): return False, {}
        if abs(ombro_e[1] - ombro_d[1]) / ombro_e[1] > TOLERANCIA_OMBRO: return False, {}
        pico1 = max(df['High'].iloc[ombro_e[0]:cabeca[0]])
        pico2 = max(df['High'].iloc[cabeca[0]:ombro_d[0]])
        neckline = (pico1 + pico2) / 2
        return True, {'tipo': 'OCO Invertido', 'altura': neckline - cabeca[1], 'neckline': neckline}

def detectar_duplo_topo(df):
    picos = encontrar_picos(df['High'], modo='max')[-2:]
    if len(picos) < 2: return False, {}
    t1, t2 = picos[-2], picos[-1]
    if abs(t1[1] - t2[1]) / t1[1] <= TOLERANCIA_NIVEL:
        vale = min(df['Low'].iloc[t1[0]:t2[0]])
        return True, {'tipo': 'Topo Duplo', 'altura': t1[1] - vale}
    return False, {}

def detectar_duplo_fundo(df):
    vales = encontrar_picos(df['Low'], modo='min')[-2:]
    if len(vales) < 2: return False, {}
    f1, f2 = vales[-2], vales[-1]
    if abs(f1[1] - f2[1]) / f1[1] <= TOLERANCIA_NIVEL:
        pico = max(df['High'].iloc[f1[0]:f2[0]])
        return True, {'tipo': 'Fundo Duplo', 'altura': pico - f1[1]}
    return False, {}

def detectar_triplo_topo(df):
    picos = encontrar_picos(df['High'], modo='max')[-3:]
    if len(picos) < 3: return False, {}
    t1, t2, t3 = picos[-3], picos[-2], picos[-1]
    if all(abs(t[1] - t1[1]) / t1[1] <= TOLERANCIA_NIVEL for t in [t1, t2, t3]):
        vale = min(min(df['Low'].iloc[t1[0]:t2[0]]), min(df['Low'].iloc[t2[0]:t3[0]]))
        return True, {'tipo': 'Topo Triplo', 'altura': t1[1] - vale}
    return False, {}

def detectar_triplo_fundo(df):
    vales = encontrar_picos(df['Low'], modo='min')[-3:]
    if len(vales) < 3: return False, {}
    f1, f2, f3 = vales[-3], vales[-2], vales[-1]
    if all(abs(f[1] - f1[1]) / f1[1] <= TOLERANCIA_NIVEL for f in [f1, f2, f3]):
        pico = max(max(df['High'].iloc[f1[0]:f2[0]]), max(df['High'].iloc[f2[0]:f3[0]]))
        return True, {'tipo': 'Fundo Triplo', 'altura': pico - f1[1]}
    return False, {}

def detectar_bandeira(df, max_dias=20):
    if len(df) < max_dias + 10: return False, {}
    preco_inicio = df['Close'].iloc[-max_dias-10]; preco_fim_mastro = df['Close'].iloc[-max_dias]
    mastro = abs(preco_fim_mastro - preco_inicio)
    if mastro / preco_inicio < 0.05: return False, {}
    alt_max = df['High'].iloc[-max_dias:].max(); alt_min = df['Low'].iloc[-max_dias:].min()
    if (alt_max - alt_min) > 0.5 * mastro: return False, {}
    direcao = 'COMPRA' if preco_fim_mastro > preco_inicio else 'VENDA'
    return True, {'tipo': 'Bandeira', 'mastro': mastro, 'direcao': direcao}

def detectar_flamula(df, max_dias=15):
    if len(df) < max_dias + 10: return False, {}
    preco_inicio = df['Close'].iloc[-max_dias-10]; preco_fim_mastro = df['Close'].iloc[-max_dias]
    mastro = abs(preco_fim_mastro - preco_inicio)
    if mastro / preco_inicio < 0.05: return False, {}
    highs = df['High'].iloc[-max_dias:]; lows = df['Low'].iloc[-max_dias:]
    picos = encontrar_picos(highs, ordem=2, modo='max'); vales = encontrar_picos(lows, ordem=2, modo='min')
    if len(picos) >= 2 and len(vales) >= 2:
        if all(picos[i][1] > picos[i+1][1] for i in range(len(picos)-1)) and all(vales[i][1] < vales[i+1][1] for i in range(len(vales)-1)):
            direcao = 'COMPRA' if preco_fim_mastro > preco_inicio else 'VENDA'
            return True, {'tipo': 'Flâmula', 'mastro': mastro, 'direcao': direcao}
    return False, {}

def detectar_cup_handle(df, max_alca_dias=14):
    if len(df) < 60: return False, {}
    highs = df['High']; topo_inicio = highs.iloc[-60:].max()
    idx_topo = highs.idxmax(); apos_topo = df.loc[idx_topo:]; fundo = apos_topo['Low'].min()
    profundidade = (topo_inicio - fundo) / topo_inicio
    if not (0.3 <= profundidade <= CUP_TOPO_CORRECAO_MAX): return False, {}
    if df['Close'].iloc[-1] < topo_inicio * 0.95: return False, {}
    alca = df.iloc[-max_alca_dias:]
    if len(alca) < 5: return False, {}
    amplitude_alca = (alca['High'].max() - alca['Low'].min()) / alca['Low'].min()
    if amplitude_alca > 0.05: return False, {}
    return True, {'tipo': 'Cup&Handle', 'altura': topo_inicio - fundo, 'resistencia': topo_inicio}

def detectar_diamante(df, min_barras=40):
    if len(df) < min_barras: return False, {}
    metade = min_barras // 2
    primeira = df.iloc[-min_barras:-metade]; segunda = df.iloc[-metade:]
    picos1 = encontrar_picos(primeira['High'], ordem=3, modo='max'); vales1 = encontrar_picos(primeira['Low'], ordem=3, modo='min')
    if len(picos1) < 2 or len(vales1) < 2: return False, {}
    if not ((picos1[-1][1] > picos1[0][1]) and (vales1[-1][1] < vales1[0][1])): return False, {}
    picos2 = encontrar_picos(segunda['High'], ordem=3, modo='max'); vales2 = encontrar_picos(segunda['Low'], ordem=3, modo='min')
    if len(picos2) < 2 or len(vales2) < 2: return False, {}
    if (picos2[-1][1] < picos2[0][1]) and (vales2[-1][1] > vales2[0][1]):
        altura = max(df['High'].iloc[-min_barras:]) - min(df['Low'].iloc[-min_barras:])
        return True, {'tipo': 'Diamante', 'altura': altura}
    return False, {}

def detectar_canal_tendencia(df, min_toques=2):
    highs = encontrar_picos(df['High'], modo='max')[-min_toques:]; lows = encontrar_picos(df['Low'], modo='min')[-min_toques:]
    if len(highs) >= min_toques and len(lows) >= min_toques:
        if all(lows[i][1] < lows[i+1][1] for i in range(len(lows)-1)):
            return True, {'tipo': 'Canal de Alta', 'direcao': 'COMPRA', 'altura': highs[-1][1] - lows[-1][1]}
        if all(highs[i][1] > highs[i+1][1] for i in range(len(highs)-1)):
            return True, {'tipo': 'Canal de Baixa', 'direcao': 'VENDA', 'altura': highs[-1][1] - lows[-1][1]}
    return False, {}

def detectar_pivo_dow(df, direcao='COMPRA'):
    if len(df) < 30: return False, {}
    lows = encontrar_picos(df['Low'], modo='min')[-3:]; highs = encontrar_picos(df['High'], modo='max')[-3:]
    if len(lows) < 2 or len(highs) < 2: return False, {}
    if direcao == 'COMPRA':
        if lows[-1][1] > lows[-2][1] and df['Close'].iloc[-1] > highs[-2][1]:
            return True, {'tipo': 'Pivô de Alta', 'altura': highs[-2][1] - lows[-2][1]}
    else:
        if highs[-1][1] < highs[-2][1] and df['Close'].iloc[-1] < lows[-2][1]:
            return True, {'tipo': 'Pivô de Baixa', 'altura': highs[-2][1] - lows[-2][1]}
    return False, {}

print("✅ Padrões gráficos carregados")

✅ Padrões gráficos carregados


In [33]:
# %% code
# ================ ESTRUTURA DE FUNDO (NOVO v16.2) ================
def detectar_rompimento_ltb(df, janela=LTB_JANELA):
    """Detecta se o preço rompeu recentemente uma LTB de longo prazo."""
    if len(df) < janela: return False, None
    trecho = df.iloc[-janela:]
    highs = encontrar_picos(trecho['High'], ordem=5, modo='max')
    if len(highs) < 2: return False, None
    p1, p2 = highs[0], highs[-1]
    if p2[1] >= p1[1]: return False, None
    slope = (p2[1] - p1[1]) / (p2[0] - p1[0])
    idx_atual = len(df) - 1
    ltb_projetada = p1[1] + slope * (idx_atual - (p1[0] + (len(df) - janela)))
    if df['Close'].iloc[-1] > ltb_projetada: return True, slope
    return False, None

def detectar_estrutura_fundo(df, rompeu_ltb):
    """Procura OCO Invertido, Fundo Duplo/Triplo acima de LTB rompida, validando volume."""
    if not rompeu_ltb: return False, None
    # Tenta detectar OCO Invertido
    ok_oco, info_oco = detectar_oco(df, 'fundo')
    if ok_oco:
        # Valida volume: seco nos fundos, explosivo no rompimento
        return True, info_oco
    # Tenta Fundo Duplo
    ok_fd, info_fd = detectar_duplo_fundo(df)
    if ok_fd: return True, info_fd
    # Tenta Fundo Triplo
    ok_ft, info_ft = detectar_triplo_fundo(df)
    if ok_ft: return True, info_ft
    return False, None

print("✅ Estrutura de Fundo carregada")

✅ Estrutura de Fundo carregada


In [34]:
# %% code
# ================ CONFLUÊNCIA UNIFICADA v16.2 ================
def calcular_score_confluencia_v16(df, direcao, entrada, setup_detectado, info_setup, volume_atual, media_volume, rsi_atual, rsi_anterior, mm20, mm50, mm200, atr, rompeu_ltb=False, estrutura_fundo=False):
    score = 0; camadas = []; vetos = []
    # Vetos
    if USAR_DIVERGENCIAS:
        if detectar_divergencia_oscilador(df, 'IFR', direcao): vetos.append('Divergência contrária IFR')
        if detectar_divergencia_oscilador(df, 'MACD', direcao): vetos.append('Divergência contrária MACD')
        if vetos: return -1, vetos, ['DIV_BLOQUEADA'], []
    if USAR_GAPS:
        tipo_gap = classificar_gap(df, atr, media_volume)
        if tipo_gap == 'Exaustão': return -1, ['Gap de Exaustão'], ['GAP_EXAUSTAO'], []
    # Pontuação base
    if setup_detectado and info_setup.get('tipo'):
        score += 25; camadas.append('padrao_grafico')
    if preco_em_zona_interesse(df, entrada, direcao):
        score += 25; camadas.append('zona_interesse')
    if direcao == 'COMPRA':
        if mm20 > mm50 > mm200 and entrada > mm20: score += 25; camadas.append('medias_alinhavadas')
    else:
        if mm20 < mm50 < mm200 and entrada < mm20: score += 25; camadas.append('medias_alinhavadas')
    if USAR_MACD_CONFLUENCIA:
        macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        if macd is not None:
            linha_macd, linha_sinal = macd['MACD_12_26_9'].iloc[-1], macd['MACDs_12_26_9'].iloc[-1]
            hist_atual, hist_ant = macd['MACDh_12_26_9'].iloc[-1], macd['MACDh_12_26_9'].iloc[-2] if len(macd)>=2 else 0
            if direcao == 'COMPRA' and linha_macd > linha_sinal and hist_atual > hist_ant: score += 20; camadas.append('macd_confluencia')
            elif direcao == 'VENDA' and linha_macd < linha_sinal and hist_atual < hist_ant: score += 20; camadas.append('macd_confluencia')
    if direcao == 'COMPRA':
        if volume_atual >= media_volume * VOLUME_MULT_MEDIO_COMPRA: score += 10; camadas.append('volume_compra')
    else:
        if volume_atual >= media_volume * VOLUME_MULT_VENDA: score += 10; camadas.append('volume_venda')
    if USAR_OBV and obv_antecipacao(df, direcao): score += 15; camadas.append('obv_antecipacao')
    if USAR_BOLLINGER and calcular_bollinger_squeeze(df): score += 20; camadas.append('bollinger_squeeze')
    if USAR_GAPS:
        tipo_gap = classificar_gap(df, atr, media_volume)
        if tipo_gap == 'Fuga': score += 20; camadas.append('gap_fuga')
    if USAR_DIVERGENCIAS:
        if detectar_divergencia_oscilador(df, 'IFR', direcao): score += 30; camadas.append('divergencia_favor')
        elif detectar_divergencia_oscilador(df, 'MACD', direcao): score += 20; camadas.append('divergencia_macd_favor')
    if info_setup.get('tipo') in ['Diamante', 'OCO Topo', 'OCO Invertido']: score += 15; camadas.append('padrao_raro')
    if direcao == 'VENDA':
        candle_atual = df.iloc[-1]; candle_ant = df.iloc[-2] if len(df)>=2 else None
        velas = classificar_candle(candle_atual, candle_ant, 'VENDA')
        if any(p in velas for p in ['Engolfo de Baixa', 'Kicker de Baixa', 'Estrela Cadente']): score += 15; camadas.append('padrao_vela_baixista')
    # Ajustes macro (v16.2)
    if USAR_FILTRO_MACRO_LTB and rompeu_ltb:
        if direcao == 'VENDA':
            score -= 30; camadas.append('penalidade_macro_ltb')
            if estrutura_fundo: score -= 15; camadas.append('penalidade_estrutura_fundo')
        elif direcao == 'COMPRA':
            score += 20; camadas.append('bonus_macro_ltb')
            if estrutura_fundo: score += 15; camadas.append('bonus_estrutura_fundo')
    return score, vetos, camadas, camadas

print("✅ Confluência unificada v16.2 carregada")


✅ Confluência unificada v16.2 carregada


In [35]:
# %% code
# ================ STOP / ALVO / PAYOFF ================
def calcular_stop_por_padrao(df, entrada, direcao, setup_nome, info_setup, atr):
    if direcao == 'COMPRA':
        if setup_nome in ['Triângulo Simétrico', 'Triângulo Ascendente']:
            ultimo_fundo = info_setup.get('fundos', [None])[-1] if 'fundos' in info_setup else None
            return ultimo_fundo - atr if ultimo_fundo else entrada - 1.8*atr
        elif setup_nome == 'Retângulo': return info_setup.get('suporte', entrada - 1.8*atr) - atr
        elif setup_nome in ['OCO Invertido', 'Fundo Duplo', 'Fundo Triplo']: return info_setup.get('fundo', info_setup.get('cabeca', 0)) - atr
        elif setup_nome in ['Bandeira', 'Flâmula']: return df['Low'].iloc[-min(20, len(df)):].min() - atr
        elif setup_nome == 'Cup&Handle': return info_setup.get('resistencia', entrada) * 0.97
        elif setup_nome in ['Canal de Alta', 'Pivô de Alta']: return df['Low'].iloc[-min(10, len(df)):].min() - atr
        else: return entrada - 1.8 * atr
    else:
        if setup_nome in ['Triângulo Descendente', 'Triângulo Simétrico']:
            ultimo_topo = info_setup.get('topos', [None])[-1] if 'topos' in info_setup else None
            return ultimo_topo + atr if ultimo_topo else entrada + 1.8*atr
        elif setup_nome == 'Retângulo': return info_setup.get('resistencia', entrada + 1.8*atr) + atr
        elif setup_nome in ['OCO Topo', 'Topo Duplo', 'Topo Triplo']: return info_setup.get('topo', info_setup.get('cabeca', 0)) + atr
        elif setup_nome in ['Bandeira', 'Flâmula']: return df['High'].iloc[-min(20, len(df)):].max() + atr
        elif setup_nome in ['Canal de Baixa', 'Pivô de Baixa']: return df['High'].iloc[-min(10, len(df)):].max() + atr
        else: return entrada + 1.8 * atr

def calcular_alvo_por_padrao(entrada, direcao, setup_nome, info_setup):
    altura = info_setup.get('altura', info_setup.get('mastro', 0))
    if altura <= 0: return None
    return entrada + altura if direcao == 'COMPRA' else entrada - altura

def calcular_payoff(entrada, alvo, stop, direcao, custos=0.003):
    risco = abs(entrada - stop)
    retorno = abs(alvo - entrada)
    if risco <= 0: return 0.0
    return round(retorno / risco - custos / risco, 2)

print("✅ Stop / Alvo / Payoff carregados")


✅ Stop / Alvo / Payoff carregados


In [36]:
# %% code
# ================ ANÁLISE PRINCIPAL v16.2 ================
def analisar_timeframe_v16(data_dict, nome_tf, tickers_liquidos, tendencia_superior=None):
    oportunidades, status, contagem_setores = [], [], {}
    for ticker in tickers_liquidos:
        df_raw = data_dict.get(ticker)
        if df_raw is None or df_raw.empty:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Sem dados'})
            continue
        df = df_raw.copy(); df.index = pd.to_datetime(df.index); df.sort_index(inplace=True)
        if len(df) < 50:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Poucos dados'})
            continue
        entrada = df['Close'].iloc[-1]
        if entrada < PRECO_MINIMO:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Preço R$ {entrada:.2f} < {PRECO_MINIMO}'})
            continue
        atr = calcular_atr(df) or entrada * 0.02
        mm20, mm50 = df['Close'].rolling(20).mean().iloc[-1], df['Close'].rolling(50).mean().iloc[-1]
        mm200 = df['Close'].rolling(200).mean().iloc[-1] if len(df)>=200 else mm50
        rsi_series = ta.rsi(df['Close'], length=14); rsi_atual = rsi_series.iloc[-1] if rsi_series is not None and len(rsi_series)>0 else 50
        media_volume, volume_atual = df['Volume'].rolling(20).mean().iloc[-1], df['Volume'].iloc[-1]

        # Contexto macro
        rompeu_ltb, _ = detectar_rompimento_ltb(df) if USAR_FILTRO_MACRO_LTB else (False, None)
        estrutura_fundo, info_fundo = detectar_estrutura_fundo(df, rompeu_ltb) if USAR_ESTRUTURA_FUNDO else (False, None)

        candidatos = []
        padroes_compra = [
            (detectar_triangulo_ascendente, 'Triângulo Ascendente'), (detectar_triangulo_simetrico, 'Triângulo Simétrico'),
            (detectar_retangulo, 'Retângulo'), (detectar_duplo_fundo, 'Fundo Duplo'), (detectar_triplo_fundo, 'Fundo Triplo'),
            (lambda x: detectar_oco(x, 'fundo'), 'OCO Invertido'), (detectar_bandeira, 'Bandeira'), (detectar_flamula, 'Flâmula'),
            (detectar_cup_handle, 'Cup&Handle'), (detectar_diamante, 'Diamante'), (detectar_canal_tendencia, 'Canal de Alta'),
            (detectar_pivo_dow, 'Pivô de Alta')
        ]
        for func, nome in padroes_compra:
            ok, info = func(df)
            if ok:
                if nome == 'Triângulo Simétrico' and df['Close'].pct_change(20).iloc[-1] <= 0: continue
                if nome == 'Diamante' and df['Close'].iloc[-1] <= df['Close'].iloc[-5]: continue
                candidatos.append((nome, info, 'COMPRA'))
        padroes_venda = [
            (detectar_triangulo_descendente, 'Triângulo Descendente'), (detectar_triangulo_simetrico, 'Triângulo Simétrico'),
            (detectar_retangulo, 'Retângulo'), (detectar_duplo_topo, 'Topo Duplo'), (detectar_triplo_topo, 'Topo Triplo'),
            (lambda x: detectar_oco(x, 'topo'), 'OCO Topo'), (detectar_bandeira, 'Bandeira'), (detectar_flamula, 'Flâmula'),
            (detectar_diamante, 'Diamante'), (detectar_canal_tendencia, 'Canal de Baixa'), (detectar_pivo_dow, 'Pivô de Baixa')
        ]
        for func, nome in padroes_venda:
            ok, info = func(df)
            if ok:
                if nome == 'Triângulo Simétrico' and df['Close'].pct_change(20).iloc[-1] >= 0: continue
                if nome == 'Diamante' and df['Close'].iloc[-1] >= df['Close'].iloc[-5]: continue
                if nome == 'Canal de Baixa' and info.get('direcao') != 'VENDA': continue
                candidatos.append((nome, info, 'VENDA'))
        if not candidatos:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': 'Nenhum padrão detectado'})
            continue

        melhor_op, melhor_score, motivos_recusa_local = None, -1, []
        for nome, info, direc in candidatos:
            candle_atual = df.iloc[-1]; candle_ant = df.iloc[-2] if len(df)>=2 else None
            ok_candle, msg_candle = validar_candle_forca_v2(candle_atual, direc)
            if not ok_candle:
                if MODO_PULLBACK:
                    linha = info.get('resistencia') or info.get('topo') if direc=='COMPRA' else info.get('suporte') or info.get('fundo')
                    if linha:
                        houve_pb, candle_pb = verificar_pullback(df, linha, direc, media_volume)
                        if houve_pb:
                            corpo = abs(candle_pb['Close'] - candle_pb['Open']); rng = candle_pb['High'] - candle_pb['Low']
                            if rng > 0 and corpo / rng >= 0.35:
                                fech_rel = (candle_pb['Close'] - candle_pb['Low']) / rng if direc=='COMPRA' else (candle_pb['High'] - candle_pb['Close']) / rng
                                if fech_rel >= 0.60: ok_candle = True; msg_candle = "OK (pullback flexibilizado)"
                if not ok_candle:
                    vol_ok = volume_atual >= (media_volume * VOLUME_MULT_MEDIO_COMPRA if direc=='COMPRA' else media_volume * VOLUME_MULT_VENDA)
                    zona_ok = preco_em_zona_interesse(df, entrada, direc)
                    if vol_ok and zona_ok:
                        motivos_recusa_local.append(f'{nome} candle: {msg_candle} (setup forte, aguardar confirmação)')
                    else:
                        motivos_recusa_local.append(f'{nome} candle: {msg_candle}')
                    continue
            vol_necessario = media_volume * VOLUME_MULT_MEDIO_COMPRA if direc=='COMPRA' else media_volume * VOLUME_MULT_VENDA
            if volume_atual < vol_necessario:
                motivos_recusa_local.append(f'{nome} volume'); continue
            entrada_real = entrada
            if MODO_PULLBACK:
                linha = info.get('resistencia') or info.get('topo') if direc=='COMPRA' else info.get('suporte') or info.get('fundo')
                if linha:
                    houve_pb, candle_pb = verificar_pullback(df, linha, direc, media_volume)
                    if houve_pb: entrada_real = candle_pb['Close']
            stop = calcular_stop_por_padrao(df, entrada_real, direc, nome, info, atr)
            alvo_padrao = calcular_alvo_por_padrao(entrada_real, direc, nome, info)
            if alvo_padrao is None: alvo_padrao = entrada_real + 3*abs(entrada_real - stop) if direc=='COMPRA' else entrada_real - 3*abs(entrada_real - stop)
            fib_vals = calcular_expansao_fibonacci(df, direc)
            fib_161 = fib_vals['161.8%'] if fib_vals else None
            alvo_final = min(alvo_padrao, fib_161) if fib_161 else alvo_padrao
            payoff = calcular_payoff(entrada_real, alvo_final, stop, direc)
            if payoff < PAYOFF_MINIMO:
                motivos_recusa_local.append(f'{nome} payoff {payoff:.1f}'); continue
            risco_percent = abs(entrada_real - stop) / entrada_real
            if not (RISCO_PERCENTUAL_MINIMO <= risco_percent <= RISCO_PERCENTUAL_MAXIMO):
                motivos_recusa_local.append(f'{nome} risco {risco_percent*100:.0f}%'); continue
            if tendencia_superior and ticker in tendencia_superior:
                tend_sup = tendencia_superior[ticker]
                if (direc == 'COMPRA' and tend_sup == 'BAIXA') or (direc == 'VENDA' and tend_sup == 'ALTA'):
                    motivos_recusa_local.append(f'{nome} tendência superior'); continue
            score, vetos, _, camadas = calcular_score_confluencia_v16(
                df, direc, entrada_real, True, info, volume_atual, media_volume, rsi_atual, rsi_atual, mm20, mm50, mm200, atr,
                rompeu_ltb, estrutura_fundo
            )
            if vetos: motivos_recusa_local.append(f'{nome} veto: {vetos}'); continue
            if score < PONTUACAO_MINIMA_CONFLUENCIA: motivos_recusa_local.append(f'{nome} score {score}'); continue
            if score > melhor_score:
                melhor_score = score
                melhor_op = {'nome': nome, 'info': info, 'direc': direc, 'entrada': entrada_real, 'stop': stop, 'alvo': alvo_final,
                             'alvo_est': fib_vals['261.8%'] if fib_vals else None, 'payoff': payoff, 'score': score, 'camadas': camadas}
        if melhor_op is None:
            motivos_str = '; '.join(list(set(motivos_recusa_local))[:3]) if motivos_recusa_local else 'Nenhum padrão passou'
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': motivos_str})
            continue
        setor = ticker.split('.')[0][:4]; contagem_setores[setor] = contagem_setores.get(setor, 0) + 1
        if contagem_setores[setor] > MAX_ATIVOS_POR_SETOR:
            status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '❌ Recusado', 'Motivo': f'Limite setor {setor}'})
            continue
        op = melhor_op
        oportunidades.append({
            'Ticker': ticker, 'Timeframe': nome_tf, 'Setup': op['nome'], 'Direcao': op['direc'],
            'Entrada': round(op['entrada'], 2), 'Stop Loss': round(op['stop'], 2), 'Alvo': round(op['alvo'], 2),
            'Alvo_Estendido': op['alvo_est'], 'Payoff': op['payoff'], 'Score': op['score'],
            'Confluencia': {'sinais': len(op['camadas']), 'detalhes': op['camadas'], 'camadas': op['camadas']},
            'Instrucao': f"[{nome_tf}] {op['nome']} – {op['direc']} em R$ {op['entrada']:.2f}, stop R$ {op['stop']:.2f}, alvo R$ {op['alvo']:.2f}. Payoff {op['payoff']:.2f}:1"
        })
        status.append({'Ticker': ticker, 'Timeframe': nome_tf, 'Status': '✅ APROVADO', 'Motivo': f'Score {op["score"]}'})
    return oportunidades, status, len(tickers_liquidos)

print("✅ Análise v16.2 carregada")

✅ Análise v16.2 carregada


In [37]:
# ==================== RELATÓRIO DETALHADO (COMPLETO v16.2) ====================
def calcular_nh_nl(data_dict):
    novas_max, novas_min, total = 0, 0, 0
    for ticker, df in data_dict.items():
        if df is None or len(df) < 52:
            continue
        try:
            max_52 = df['High'].rolling(52).max().iloc[-1]
            min_52 = df['Low'].rolling(52).min().iloc[-1]
            close = df['Close'].iloc[-1]
            if pd.notna(max_52) and pd.notna(min_52):
                total += 1
                if close >= max_52 * 0.995:
                    novas_max += 1
                if close <= min_52 * 1.005:
                    novas_min += 1
        except:
            pass
    saldo = novas_max - novas_min
    pct_max = round(novas_max/total*100,1) if total>0 else 0
    pct_min = round(novas_min/total*100,1) if total>0 else 0
    if saldo > 20:
        diag = "🟢 FORTE (Tendência de alta consistente)"
    elif saldo > 0:
        diag = "🟡 NEUTRO/POSITIVO (Alta moderada)"
    elif saldo > -20:
        diag = "🟠 NEUTRO/NEGATIVO (Baixa moderada)"
    else:
        diag = "🔴 FRACO (Tendência de baixa acentuada)"
    return {'saldo': saldo, 'max': novas_max, 'min': novas_min, 'total': total,
            'pct_max': pct_max, 'pct_min': pct_min, 'diagnostico': diag}

def calcular_regime_volatilidade():
    try:
        ibov = yf.download("^BVSP", period="3mo", interval="1d", progress=False)
        if ibov.empty:
            return "N/D"
        ret = ibov['Close'].pct_change().dropna()
        vol_atual = ret.tail(20).std()
        vol_hist = ret.rolling(50).std().mean()
        if vol_atual > vol_hist * 1.2:
            return "ALTA 🚀"
        elif vol_atual < vol_hist * 0.8:
            return "BAIXA 🐢"
        else:
            return "NORMAL ⚖️"
    except:
        return "N/D"

def percentual_acima_media_200(data_dict):
    acima, total = 0, 0
    for ticker, df in data_dict.items():
        if df is None or len(df) < 200:
            continue
        mm200 = df['Close'].rolling(200).mean().iloc[-1]
        if pd.notna(mm200) and df['Close'].iloc[-1] > mm200:
            acima += 1
        total += 1
    return round(acima/total*100,1) if total>0 else 0

def gerar_relatorio_detalhado(ops, todos_status, data_dict, nome_tf="Diário"):
    nhnl = calcular_nh_nl(data_dict)
    regime = calcular_regime_volatilidade()
    pct_acima_200 = percentual_acima_media_200(data_dict)

    recusados = [s for s in todos_status if 'Recusado' in s.get('Status', '')]
    motivos = Counter([s.get('Motivo', 'Desconhecido') for s in recusados])
    top_motivos = motivos.most_common(10)

    watchlist = []
    for s in recusados:
        motivo = s.get('Motivo', '')
        if 'setup forte, aguardar confirmação' in motivo and 'Payoff' not in motivo:
            watchlist.append(f"{s['Ticker']} ({s.get('Timeframe', '?')}) – {motivo[:120]}")
    watchlist = list(dict.fromkeys(watchlist))[:15]

    linhas = []
    linhas.append("=" * 90)
    linhas.append(f"   📊 RELATÓRIO GEBRA v16.2 – ARQUITETURA COMPLETA COM ESTRUTURA DE FUNDO")
    linhas.append(f"   {datetime.now().strftime('%d/%m/%Y %H:%M')} | Timeframe principal: {nome_tf}")
    linhas.append("=" * 90)
    linhas.append("")

    linhas.append("## 🧭 1. CONDIÇÃO DE MERCADO")
    linhas.append(f"   • Regime de volatilidade (Ibovespa): {regime}")
    linhas.append(f"   • Ativos acima da MME200: {pct_acima_200}%")
    linhas.append(f"   • NH‑NL (52 semanas): saldo {nhnl['saldo']} (Máx: {nhnl['max']} / Mín: {nhnl['min']} / Total: {nhnl['total']})")
    linhas.append(f"   • Diagnóstico NH‑NL: {nhnl['diagnostico']}")
    if pct_acima_200 > 60:
        linhas.append("   ➜ Mercado em tendência de ALTA (maioria dos ativos acima da MME200)")
    elif pct_acima_200 < 40:
        linhas.append("   ➜ Mercado em tendência de BAIXA (maioria abaixo da MME200)")
    else:
        linhas.append("   ➜ Mercado lateral / indefinido")
    linhas.append("")

    linhas.append("## ✅ 2. OPORTUNIDADES CONFIRMADAS")
    if ops:
        for i, op in enumerate(ops[:MAX_SETUPS_POR_DIA], 1):
            linhas.append(f"   {i}. [{op['Timeframe']}] {op['Ticker']} | {op['Setup']} | {op['Direcao']}")
            linhas.append(f"      Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f} | Alvo: R$ {op['Alvo']:.2f}")
            linhas.append(f"      Payoff: {op['Payoff']:.2f} | Score: {op['Score']} | Sinais: {len(op['Confluencia']['camadas'])}")
            linhas.append(f"      Camadas: {', '.join(op['Confluencia']['camadas'])}")
            if op.get('Alvo_Estendido'):
                linhas.append(f"      Alvo estendido (261.8%): R$ {op['Alvo_Estendido']:.2f}")
            linhas.append("")
    else:
        linhas.append("   ❌ Nenhuma oportunidade com confluência total foi encontrada.")
        linhas.append("      Possíveis causas: mercado lateral, falta de volume, candle sem convicção.")
        linhas.append("")

    linhas.append("## 🔍 3. PRINCIPAIS MOTIVOS DE RECUSA")
    if top_motivos:
        for motivo, qtd in top_motivos:
            linhas.append(f"   • {motivo}: {qtd} ocorrências")
    else:
        linhas.append("   Nenhum ativo recusado (todos aprovados ou sem dados).")
    linhas.append("")

    linhas.append("## 👀 4. WATCHLIST (SETUPS FORTES AGUARDANDO CONFIRMAÇÃO)")
    if watchlist:
        for i, item in enumerate(watchlist, 1):
            linhas.append(f"   {i}. {item}")
    else:
        linhas.append("   Nenhum setup forte aguardando confirmação no momento.")
    linhas.append("")

    linhas.append("## 💡 5. RECOMENDAÇÃO DO ARQUITETO")
    if nhnl['saldo'] > 10 and pct_acima_200 > 55 and regime != "ALTA 🚀":
        linhas.append("   ✅ Ambiente favorável para operações compradas. Busque rompimentos de resistência com volume e estruturas de fundo.")
    elif nhnl['saldo'] < -10 and pct_acima_200 < 45:
        linhas.append("   ⚠️ Tendência de baixa. Priorize operações vendidas ou mantenha-se em caixa.")
    elif regime == "ALTA 🚀":
        linhas.append("   ⚠️ Volatilidade alta! Reduza o tamanho das posições e use stops mais largos (ATR×2).")
    else:
        linhas.append("   🟡 Mercado sem direção clara. Aguarde confirmação de tendência ou opere apenas setups de alta confluência (mínimo 4 sinais).")
    linhas.append("   🔬 Filtro macro ativo: vendas contra rompimento de LTB são penalizadas; compras em pullback recebem bônus.")
    linhas.append("")

    linhas.append("---")
    linhas.append(f"   📅 Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    linhas.append(f"   🔧 Log detalhado: {ARQUIVO_LOG}")
    return "\n".join(linhas)

# Gerar relatório
relatorio = gerar_relatorio_detalhado(todas_oportunidades, todos_status, data_diario, nome_tf="Diário")
with open('relatorio_gebra_v16_2.txt', 'w', encoding='utf-8') as f:
    f.write(relatorio)

# Enviar e-mail
def enviar_email(assunto, corpo):
    if not EMAIL_REMETENTE or not SENHA_APP:
        logger.log("E‑mail não configurado – relatório salvo localmente")
        return
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_REMETENTE
        msg['To'] = EMAIL_REMETENTE
        msg['Subject'] = assunto
        msg.attach(MIMEText(corpo, 'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as srv:
            srv.login(EMAIL_REMETENTE, SENHA_APP)
            srv.send_message(msg)
        logger.log("📧 E‑mail enviado com relatório detalhado")
    except Exception as e:
        _log_exc('Email', e)

enviar_email(
    f"📈 GEBRA v16.2 - {len(todas_oportunidades)} ops - {datetime.now().strftime('%d/%m %H:%M')}",
    relatorio
)

# Telegram
def mensagem_telegram(ops):
    if not ops:
        return "📊 GEBRA v16.2\nNenhuma oportunidade.\nVerifique o relatório completo no e‑mail."
    msg = f"🚀 GEBRA v16.2 - {datetime.now().strftime('%d/%m %H:%M')}\n\n"
    for i, op in enumerate(ops[:5], 1):
        msg += f"{i}. <b>[{op['Timeframe']}] {op['Ticker']}</b> | {op['Setup']} | {op['Direcao']}\n"
        msg += f"   Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f} | Alvo: R$ {op['Alvo']:.2f} | Payoff: {op['Payoff']:.2f}\n\n"
    return msg

enviar_telegram(mensagem_telegram(todas_oportunidades))

# Log JSON detalhado
try:
    aprovados = [s for s in todos_status if 'APROVADO' in s.get('Status', '')]
    recusados = [s for s in todos_status if 'Recusado' in s.get('Status', '')]
    motivos_recusa = Counter([s.get('Motivo', '') for s in recusados])
    log_json = {
        'timestamp': datetime.now().isoformat(),
        'versao': 'v16.2',
        'status_final': 'concluido',
        'metricas': {
            'total_analisados': len(todos_status),
            'aprovados': len(aprovados),
            'recusados': len(recusados),
            'oportunidades_finais': len(todas_oportunidades)
        },
        'nhnl': calcular_nh_nl(data_diario),
        'regime_volatilidade': calcular_regime_volatilidade(),
        'percentual_acima_mm200': percentual_acima_media_200(data_diario),
        'resumo_recusas': dict(motivos_recusa.most_common(10)),
        'watchlist': [{'ticker': s['Ticker'], 'timeframe': s.get('Timeframe', ''), 'motivo': s.get('Motivo', '')}
                      for s in recusados if 'setup forte' in s.get('Motivo', '')][:15],
        'oportunidades': todas_oportunidades
    }
    with open(ARQUIVO_LOG, 'w', encoding='utf-8') as f:
        json.dump(log_json, f, indent=2, default=str)
    logger.log("📄 Log JSON detalhado salvo")
except Exception as e:
    _log_exc('log_json', e)

gc.collect()
logger.resumo()
print("\n✅ Análise concluída. Relatório: relatorio_gebra_v16_2.txt")
print(f"   Oportunidades: {len(todas_oportunidades)}")

[00:56:06] [ETAPA] 🚀 INÍCIO: Verificação de Conectividade
[00:56:06] [ETAPA] ✅ FIM: Verificação de Conectividade (0.0s)
[00:56:06] [ETAPA] 🚀 INÍCIO: Coleta de Tickers
[00:56:06] [INFO] 📦 Cache tickers (391 ativos)
[00:56:06] [ETAPA] ✅ FIM: Coleta de Tickers (0.0s) | total:391
[00:56:06] [ETAPA] 🚀 INÍCIO: Download Histórico (Diário) – via Brapi
[00:56:06] [INFO] Fallback yfinance...
[00:56:06] [INFO] 🔄 yfinance v3: 1/391
[00:56:19] [INFO] 🔄 yfinance v3: 21/391
[00:56:31] [INFO] 🔄 yfinance v3: 41/391
[00:56:41] [INFO] 🔄 yfinance v3: 61/391
[00:56:52] [INFO] 🔄 yfinance v3: 81/391
[00:57:03] [INFO] 🔄 yfinance v3: 101/391
[00:57:14] [INFO] 🔄 yfinance v3: 121/391
[00:57:25] [INFO] 🔄 yfinance v3: 141/391
[00:57:35] [INFO] 🔄 yfinance v3: 161/391
[00:57:46] [INFO] 🔄 yfinance v3: 181/391
[00:57:56] [INFO] 🔄 yfinance v3: 201/391
[00:58:07] [INFO] 🔄 yfinance v3: 221/391
[00:58:18] [INFO] 🔄 yfinance v3: 241/391


ERROR:yfinance:$AZEV11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$AZEV11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$AZEV11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$AZEV11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$AZEV11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$AZEV11: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


[00:58:39] [INFO] 🔄 yfinance v3: 261/391
[00:58:50] [INFO] 🔄 yfinance v3: 281/391
[00:59:01] [INFO] 🔄 yfinance v3: 301/391
[00:59:12] [INFO] 🔄 yfinance v3: 321/391
[00:59:22] [INFO] 🔄 yfinance v3: 341/391


ERROR:yfinance:$BIOM11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$BIOM11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$BIOM11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$BIOM11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$BIOM11.SA: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$BIOM11: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


[00:59:43] [INFO] 🔄 yfinance v3: 361/391
[00:59:53] [INFO] 🔄 yfinance v3: 381/391
[00:59:58] [INFO] 📦 yfinance v3: 385 tickers
[00:59:58] [ETAPA] 🚀 INÍCIO: Filtro de Liquidez
[00:59:58] [ETAPA] ✅ FIM: Filtro de Liquidez (0.2s) | liquidos:137
[01:06:06] [RESULTADO] 🎯 Total oportunidades finais: 1
[01:06:07] [RESUMO] 
[01:06:07] [RESUMO] ⏱️ Tempo total: 601.5s
[01:06:07] [RESUMO]    • Verificação de Conectividade: 0.0s (0%)
[01:06:07] [RESUMO]    • Coleta de Tickers: 0.0s (0%)
[01:06:07] [RESUMO]    • Filtro de Liquidez: 0.2s (0%)
[01:06:07] [RESUMO] ============================================================


✅ Análise concluída. Relatório: relatorio_gebra_v16_2.txt | Oportunidades: 1
